# Multimodal Content Moderation with Vision-Language Models (CLIP)
### Mini Project — AI Minor | Meme Classification & Harmful Content Detection

**What this notebook does:**
Detects and classifies harmful or offensive image+text memes using OpenAI's **pretrained CLIP (Contrastive Language-Image Pretraining)** vision-language model as a multimodal feature extractor, paired with a trained classifier head.

**Key Highlights:**
1. **Multimodal Fusion**: Combines visual features (ViT) and semantic textual features into a 1024-dimensional joint representation.
2. **Benchmark Comparison**: Evaluates against a text-only TF-IDF baseline to prove the value of multimodal perception.
3. **Class Balancing & Binary Moderation**: Implements both binary content moderation (`not_offensive` vs `offensive`) and 4-tier severity levels.
4. **Visual Error Analysis**: Side-by-side inspection of edge cases (irony, sarcasm, multimodal dissonance).
5. **Interactive Gradio Web Demo**: Built-in interactive demo for testing custom memes in real-time.
6. **Architecture & Design Rationale**: In-depth analysis of multimodal fusion decisions and failure modes.

**Colab Instructions:**
* Go to **Runtime → Change runtime type → GPU (T4)** for fast embedding extraction.
* Obtain a free Kaggle API token (`kaggle.json`) from [kaggle.com/settings](https://www.kaggle.com/settings).


## 1. Setup & Environment
Install required libraries, verify GPU availability, and mount Google Drive for persistent storage.


In [ ]:
# Install dependencies
!pip install -q transformers torch torchvision scikit-learn pandas numpy matplotlib seaborn gradio kagglehub pillow joblib tqdm


In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from tqdm.auto import tqdm

# Allow loading truncated images to avoid corrupt file crashes
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

import torch
import torchvision

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Consider enabling GPU (Runtime > Change runtime type > T4 GPU) for 10x faster feature extraction.")


### 1.1 Connect Google Drive (Method A: Persistent Cloud Sync)
Mount your Google Drive so the downloaded dataset, checkpoints, and final model (`clip_classifier.pkl`) are saved directly in your Google Drive under `Meme Classification`.


In [ ]:
# Connect Google Drive (runs seamlessly on the same Google account)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set up dedicated project directory in Google Drive
    drive_project_dir = '/content/drive/MyDrive/Meme Classification'
    os.makedirs(drive_project_dir, exist_ok=True)
    os.chdir(drive_project_dir)
    print(f"[+] Successfully connected to Google Drive!")
    print(f"[+] Active working directory: {os.getcwd()}")
except ImportError:
    print("[*] Running in local environment (Google Drive mount not applicable).")
except Exception as e:
    print(f"[!] Drive mount note: {e}")


## 2. Dataset Acquisition (Memotion 7k)
Download the Memotion 7k dataset automatically using `kagglehub`.


In [ ]:
import kagglehub

# Authenticate Kaggle account (enter username and key when prompted, or upload kaggle.json)
kagglehub.login()


In [ ]:
# Download Memotion 7k dataset
print("Downloading Memotion 7k dataset...")
dataset_path = kagglehub.dataset_download("williamscott701/memotion-dataset-7k")
print(f"Dataset downloaded to: {dataset_path}")


In [ ]:
# Automatically resolve dataset directory structure
possible_csv_paths = [
    os.path.join(dataset_path, "memotion_dataset_7k", "labels.csv"),
    os.path.join(dataset_path, "labels.csv")
]

possible_img_dirs = [
    os.path.join(dataset_path, "memotion_dataset_7k", "images"),
    os.path.join(dataset_path, "images")
]

LABELS_CSV = next((p for p in possible_csv_paths if os.path.exists(p)), None)
IMAGES_DIR = next((p for p in possible_img_dirs if os.path.isdir(p)), None)

if LABELS_CSV is None or IMAGES_DIR is None:
    # Fallback search if directory layout is unexpected
    for root, dirs, files in os.walk(dataset_path):
        if "labels.csv" in files and LABELS_CSV is None:
            LABELS_CSV = os.path.join(root, "labels.csv")
        if "images" in dirs and IMAGES_DIR is None:
            IMAGES_DIR = os.path.join(root, "images")

print(f"Labels CSV found at : {LABELS_CSV}")
print(f"Images folder found at: {IMAGES_DIR}")

# Verify file counts
image_files = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Total valid images on disk: {len(image_files)}")


## 3. Data Cleaning & Problem Formulation

### Binary Content Moderation vs. Fine-Grained Severity
The Memotion dataset contains 4 offensive levels:
- `not_offensive`
- `slight`
- `very_offensive`
- `hateful_offensive`

In real-world Trust & Safety pipelines, content moderation begins with **Binary Triage**:
- Class 0: **Not Offensive** (`not_offensive`)
- Class 1: **Offensive / Harmful** (`slight`, `very_offensive`, `hateful_offensive`)

Collapsing severity into a binary target provides a balanced problem and eliminates extreme class scarcity (where `hateful_offensive` has only ~150 examples). We formulate both targets so we can train and evaluate both!


In [ ]:
# Read dataset
df_raw = pd.read_csv(LABELS_CSV)
print("Raw dataset shape:", df_raw.shape)

# Identify key columns
TEXT_COL = "text_corrected" if "text_corrected" in df_raw.columns else "text_ocr"
IMAGE_COL = "image_name"
LABEL_COL = "offensive"

# Keep essential columns and drop missing values
df = df_raw[[IMAGE_COL, TEXT_COL, LABEL_COL]].dropna().copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()

# Verify which images actually exist on disk to prevent runtime IO errors
existing_images = set(os.listdir(IMAGES_DIR))
df = df[df[IMAGE_COL].isin(existing_images)].reset_index(drop=True)
print(f"Verified rows with images on disk: {len(df)}")

# Create Binary Target (0: Not Offensive, 1: Offensive)
df["binary_label"] = df[LABEL_COL].apply(lambda x: "not_offensive" if str(x).strip() == "not_offensive" else "offensive")

print("
--- 4-Class Distribution ---")
print(df[LABEL_COL].value_counts())

print("
--- Binary Moderation Distribution ---")
print(df["binary_label"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

# Split into Train (70%), Val (15%), Test (15%) stratified on the binary label
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["binary_label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["binary_label"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train set: {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val set  : {len(val_df)} samples ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test set : {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")


## 4. Baseline: Text-Only Classifier (TF-IDF + Logistic Regression)

Before applying multimodal vision-language models, we establish a **text-only baseline**.
Memes often use harmless text that is made offensive by the image, or vice-versa. If our multimodal model cannot outperform this baseline, it indicates either poor multimodal integration or reliance solely on textual priors.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

# Fit TF-IDF Vectorizer on meme text
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
X_train_text = tfidf.fit_transform(train_df[TEXT_COL])
X_val_text = tfidf.transform(val_df[TEXT_COL])
X_test_text = tfidf.transform(test_df[TEXT_COL])

# Train Logistic Regression Baseline with balanced weights
baseline_clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
baseline_clf.fit(X_train_text, train_df["binary_label"])

# Evaluate Baseline
baseline_preds = baseline_clf.predict(X_test_text)
baseline_acc = accuracy_score(test_df["binary_label"], baseline_preds)
baseline_f1 = f1_score(test_df["binary_label"], baseline_preds, average="macro")

print("=== Text-Only Baseline Performance ===")
print(f"Accuracy: {baseline_acc:.4f}")
print(f"Macro F1: {baseline_f1:.4f}
")
print(classification_report(test_df["binary_label"], baseline_preds))


## 5. The Multimodal Architecture: CLIP Feature Extraction

### Why OpenAI CLIP (`clip-vit-base-patch32`)?
1. **Zero-Shot & Shared Embedding Space**: CLIP maps both images and natural language text into a shared 512-dimensional vector space trained on 400M (image, text) pairs.
2. **Visual & Semantic Context**:
   - Image embedding $v_i \in \mathbb{R}^{512}$ captures visual tropes, expressions, and symbols.
   - Text embedding $v_t \in \mathbb{R}^{512}$ captures meme captions and wordplay.
3. **Multimodal Fusion**: We $L_2$-normalize each vector and concatenate them into a unified 1024-dimensional feature vector:
   $$v_{multimodal} = [ \frac{v_i}{\|v_i\|}, \frac{v_t}{\|v_t\|} ] \in \mathbb{R}^{1024}$$
4. **Frozen Backbone + Trainable Head**: Freezing CLIP avoids overfitting on the small 7k dataset while allowing rapid, compute-efficient training of a classifier head.


In [ ]:
from transformers import CLIPModel, CLIPProcessor

model_name = "openai/clip-vit-base-patch32"
print(f"Loading pretrained CLIP model: {model_name}...")

clip_model = CLIPModel.from_pretrained(model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(model_name)
clip_model.eval()  # Freeze backbone for feature extraction
print("CLIP loaded successfully!")


In [ ]:
@torch.no_grad()
def extract_clip_features_batch(df_subset, images_dir, batch_size=32):
    """
    Batched CLIP feature extraction for high throughput on GPU.
    Returns: (features_array, valid_indices)
    """
    features_list = []
    valid_indices = []

    for start_idx in tqdm(range(0, len(df_subset), batch_size), desc="Extracting CLIP features"):
        end_idx = min(start_idx + batch_size, len(df_subset))
        batch_df = df_subset.iloc[start_idx:end_idx]

        batch_images = []
        batch_texts = []
        batch_valid = []

        for idx, row in batch_df.iterrows():
            img_path = os.path.join(images_dir, row[IMAGE_COL])
            try:
                # Open and convert to RGB
                img = Image.open(img_path).convert("RGB")
                text = str(row[TEXT_COL]) if str(row[TEXT_COL]).strip() else "meme"
                batch_images.append(img)
                batch_texts.append(text)
                batch_valid.append(idx)
            except Exception as e:
                # Silently skip corrupt images
                continue

        if not batch_images:
            continue

        # Process batch through CLIP (max_length=77 handles CLIP tokenizer limit)
        inputs = clip_processor(
            text=batch_texts,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        ).to(device)

        # Forward pass
        outputs = clip_model(**inputs)

        # Get normalized embeddings
        img_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
        text_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)

        # Multimodal concatenation (512 + 512 = 1024)
        fused_embeds = torch.cat([img_embeds, text_embeds], dim=-1).cpu().numpy()

        features_list.append(fused_embeds)
        valid_indices.extend(batch_valid)

    if features_list:
        return np.vstack(features_list), valid_indices
    return np.empty((0, 1024)), []


In [ ]:
# Extract Multimodal Features for Train, Val, and Test splits
# (On a T4 GPU, ~4800 images takes under 90 seconds)
print("Extracting train set features...")
X_train, train_valid_idx = extract_clip_features_batch(train_df, IMAGES_DIR, batch_size=32)
y_train = train_df.loc[train_valid_idx, "binary_label"].values
train_df_clean = train_df.loc[train_valid_idx].reset_index(drop=True)

print("Extracting val set features...")
X_val, val_valid_idx = extract_clip_features_batch(val_df, IMAGES_DIR, batch_size=32)
y_val = val_df.loc[val_valid_idx, "binary_label"].values
val_df_clean = val_df.loc[val_valid_idx].reset_index(drop=True)

print("Extracting test set features...")
X_test, test_valid_idx = extract_clip_features_batch(test_df, IMAGES_DIR, batch_size=32)
y_test = test_df.loc[test_valid_idx, "binary_label"].values
test_df_clean = test_df.loc[test_valid_idx].reset_index(drop=True)

print(f"X_train shape: {X_train.shape}, y_train count: {len(y_train)}")
print(f"X_test shape : {X_test.shape}, y_test count: {len(y_test)}")


## 6. Train the Multimodal Classifier Heads

We evaluate two standard classifier heads on top of the fixed CLIP embeddings:
1. **Logistic Regression Probe (Linear Probe)**: Strong, interpretable, resistant to overfitting.
2. **Multi-Layer Perceptron (MLPClassifier)**: Non-linear head capturing non-linear cross-modal interactions.


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder

# Encode string labels
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)
classes = list(label_encoder.classes_)
print(f"Classes: {classes}")

# 1. Linear Probe (Logistic Regression)
clip_lr = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0, random_state=42)
clip_lr.fit(X_train, y_train_enc)
lr_preds_enc = clip_lr.predict(X_test)
lr_acc = accuracy_score(y_test_enc, lr_preds_enc)
lr_f1 = f1_score(y_test_enc, lr_preds_enc, average="macro")

# 2. Non-linear MLP Head
clip_mlp = MLPClassifier(
    hidden_layer_sizes=(256, 64),
    activation="relu",
    max_iter=300,
    alpha=1e-4,
    early_stopping=True,
    n_iter_no_change=10,
    random_state=42
)
clip_mlp.fit(X_train, y_train_enc)
mlp_preds_enc = clip_mlp.predict(X_test)
mlp_acc = accuracy_score(y_test_enc, mlp_preds_enc)
mlp_f1 = f1_score(y_test_enc, mlp_preds_enc, average="macro")

print(f"=== Multimodal Results on Test Set ===")
print(f"Linear Probe (LR) -> Accuracy: {lr_acc:.4f} | Macro F1: {lr_f1:.4f}")
print(f"MLP Head          -> Accuracy: {mlp_acc:.4f} | Macro F1: {mlp_f1:.4f}")

# Select the best performing model
best_model = clip_lr if lr_f1 >= mlp_f1 else clip_mlp
best_model_name = "Logistic Regression (Linear Probe)" if lr_f1 >= mlp_f1 else "MLP Classifier"
best_preds_enc = lr_preds_enc if lr_f1 >= mlp_f1 else mlp_preds_enc
best_acc = max(lr_acc, mlp_acc)
best_f1 = max(lr_f1, mlp_f1)
best_preds = label_encoder.inverse_transform(best_preds_enc)

print(f"
Selected Best Model: {best_model_name}")
print(f"Improvement over Text Baseline: {best_f1 - baseline_f1:+.4f} Macro F1")
print("
" + classification_report(y_test, best_preds))


## 7. Evaluation: Confusion Matrices & Comparison Charts


In [ ]:
# Confusion Matrices comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_base = confusion_matrix(test_df["binary_label"], baseline_preds, labels=classes)
cm_clip = confusion_matrix(y_test, best_preds, labels=classes)

sns.heatmap(cm_base, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=classes, yticklabels=classes)
axes[0].set_title(f"Baseline (Text-Only TF-IDF)\nAcc: {baseline_acc:.3f} | Macro F1: {baseline_f1:.3f}")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_clip, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=classes, yticklabels=classes)
axes[1].set_title(f"Multimodal CLIP + {best_model_name}\nAcc: {best_acc:.3f} | Macro F1: {best_f1:.3f}")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

# Performance comparison bar chart
metrics_df = pd.DataFrame({
    "Model": ["Text Baseline (TF-IDF)", f"Multimodal CLIP ({best_model_name})"],
    "Accuracy": [baseline_acc, best_acc],
    "Macro F1": [baseline_f1, best_f1]
})

plt.figure(figsize=(7, 4))
df_melt = metrics_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
sns.barplot(data=df_melt, x="Metric", y="Score", hue="Model", palette=["#90caf9", "#4caf50"])
plt.ylim(0, 1.0)
plt.title("Performance Comparison: Text-Only vs. Multimodal CLIP")
for p in plt.gca().patches:
    val = p.get_height()
    if val > 0:
        plt.gca().annotate(f"{val:.2f}", (p.get_x() + p.get_width() / 2., val / 2),
                           ha='center', va='center', color='white', fontweight='bold')
plt.show()


## 8. Qualitative Error Analysis
Visual inspection of memes where:
1. Multimodal succeeded while Text-Only failed (demonstrating why visual cues matter).
2. Edge cases where both models struggled (subtle sarcasm, cultural references).


In [ ]:
# Identify test indices where CLIP was correct but Baseline was wrong
correct_multimodal = (best_preds == y_test)
wrong_baseline = (baseline_preds == y_test)
multimodal_advantage_idx = np.where(correct_multimodal & (~wrong_baseline))[0]

print(f"Total memes where Multimodal succeeded but Text Baseline failed: {len(multimodal_advantage_idx)}")

# Display up to 3 showcase examples
show_count = min(3, len(multimodal_advantage_idx))
if show_count > 0:
    fig, axes = plt.subplots(1, show_count, figsize=(5 * show_count, 5))
    if show_count == 1:
        axes = [axes]

    for i in range(show_count):
        idx = multimodal_advantage_idx[i]
        row = test_df_clean.iloc[idx]
        img_path = os.path.join(IMAGES_DIR, row[IMAGE_COL])

        try:
            img = Image.open(img_path)
            axes[i].imshow(img)
            axes[i].axis("off")
            caption = f"True: {row['binary_label']}\nCLIP: {best_preds[idx]} (Correct)\nText-Only: {baseline_preds[idx]} (Wrong)\nText: {row[TEXT_COL][:60]}..."
            axes[i].set_title(caption, fontsize=9)
        except Exception as e:
            axes[i].text(0.5, 0.5, "Image load error", ha="center")

    plt.tight_layout()
    plt.show()


## 9. Explainability: Cross-Modal Alignment Score
We can analyze the **Image-Text Cosine Similarity** computed directly by CLIP:
$$\text{Alignment} = \cos(v_i, v_t) = \frac{v_i \cdot v_t}{\|v_i\| \|v_t\|}$$
In non-meme datasets, high alignment means the text literally describes the image. In memes, low alignment often signals **metaphor, irony, or multimodal dissonance**.


In [ ]:
# Compute image-text similarity distribution across categories
alignment_scores = []

for idx in range(min(500, len(X_test))):
    # Image features are first 512, text features are last 512
    v_img = X_test[idx, :512]
    v_txt = X_test[idx, 512:]
    sim = np.dot(v_img, v_txt)
    alignment_scores.append({
        "similarity": float(sim),
        "label": y_test[idx]
    })

sim_df = pd.DataFrame(alignment_scores)

plt.figure(figsize=(7, 4))
sns.kdeplot(data=sim_df, x="similarity", hue="label", fill=True, common_norm=False, palette="Set2")
plt.title("CLIP Image-Text Cross-Modal Alignment Score Distribution")
plt.xlabel("Cosine Similarity between Image and Text Embeddings")
plt.show()


## 10. Save Model Artifacts for Web Demo (`app.py`)
Serialize the trained model, label encoder, and configuration into `clip_classifier.pkl` in your Google Drive or current folder so our Gradio app can perform instant inference without retraining.


In [ ]:
import joblib

# Export model bundle
model_bundle = {
    "classifier": best_model,
    "model_name": best_model_name,
    "label_encoder": label_encoder,
    "classes": list(label_encoder.classes_),
    "clip_model_name": "openai/clip-vit-base-patch32",
    "feature_dim": 1024,
    "target": "binary_content_moderation"
}

artifact_path = "clip_classifier.pkl"
joblib.dump(model_bundle, artifact_path)
print(f"Successfully saved complete model bundle to: {os.path.abspath(artifact_path)}")
print(f"File size: {os.path.getsize(artifact_path) / 1024:.1f} KB")
print("\nThis model bundle is saved directly in your Google Drive/project directory for local or hosted deployment!")


## 11. Interactive Web Demo (Live in Notebook)
Test any meme directly inside the notebook with Gradio!


In [ ]:
import gradio as gr

def predict_meme(image, meme_text):
    if image is None:
        return "Please upload an image.", {}

    text_input = str(meme_text).strip() if meme_text else "meme"

    # Convert PIL Image to RGB
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert("RGB")
    else:
        image = image.convert("RGB")

    # Extract CLIP Embeddings
    inputs = clip_processor(
        text=[text_input],
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    ).to(device)

    with torch.no_grad():
        outputs = clip_model(**inputs)
        img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
        txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
        fused = torch.cat([img_emb, txt_emb], dim=-1).cpu().numpy()

    # Predict with trained classifier head
    if hasattr(best_model, "predict_proba"):
        probs = best_model.predict_proba(fused)[0]
    else:
        # Decision function fallback
        decision = best_model.decision_function(fused)[0]
        prob_1 = 1 / (1 + np.exp(-decision))
        probs = [1 - prob_1, prob_1]

    pred_idx = np.argmax(probs)
    pred_label = classes[pred_idx]
    confidence_dict = {classes[i].replace('_', ' ').title(): float(probs[i]) for i in range(len(classes))}

    status = "⚠️ FLAGGED AS OFFENSIVE / HARMFUL" if pred_label == "offensive" else "✅ APPROVED (NOT OFFENSIVE)"
    summary = f"### {status}\n**Predicted Class:** {pred_label.replace('_', ' ').title()}\n**Confidence:** {probs[pred_idx]*100:.1f}%"

    return summary, confidence_dict

# Launch Gradio interface
demo = gr.Interface(
    fn=predict_meme,
    inputs=[
        gr.Image(type="pil", label="Upload Meme Image"),
        gr.Textbox(label="Meme Text (Caption / Overlay OCR)", placeholder="Type the meme caption here...")
    ],
    outputs=[
        gr.Markdown(label="Moderation Verdict"),
        gr.Label(label="Class Probabilities")
    ],
    title="🛡️ Multimodal Meme Moderation System (CLIP)",
    description="Upload a meme image and enter its text overlay to classify whether it violates content safety policies.",
    theme="soft"
)

# Launch (share=True creates a public temporary link for Colab)
demo.launch(share=False, debug=False)


## 12. Architecture & Design Rationale 💡

Key engineering decisions and technical takeaways from this project:

### Q1: Why use a Multimodal Vision-Language Model (CLIP) instead of separate image and text models?
* **Answer:** Memes rely on **multimodal interaction**. The text alone may be harmless (e.g., "Look at this creature"), and the image alone may be harmless (a picture of an animal). However, juxtaposing them can produce hateful or derogatory meaning. CLIP aligns visual and textual tokens into a shared semantic space, enabling the classifier to capture this cross-modal relationship.

### Q2: Why freeze the CLIP backbone and only train a classifier head (linear probe / MLP)?
* **Answer:**
  1. **Compute & Efficiency:** Fine-tuning the full 150M parameter CLIP backbone requires substantial VRAM and training time.
  2. **Avoiding Catastrophic Forgetting & Overfitting:** With only ~7,000 examples, fine-tuning the entire backbone would cause severe overfitting. CLIP's representations are already pre-trained on 400M pairs.

### Q3: Why did you collapse the 4 Memotion classes into binary labels?
* **Answer:** In real-world content moderation (e.g., Meta, Reddit), initial automated filters perform binary triage (safe vs. unsafe). Furthermore, the 4-class Memotion dataset suffered from acute class imbalance (`hateful_offensive` had only ~150 samples / 2% of data), resulting in near-zero recall. Binary collapsing provides balanced statistical power.

### Q4: How is the multimodal feature vector constructed?
* **Answer:**
  1. Image passed through CLIP ViT $\rightarrow$ normalized 512-dim vector $v_i$.
  2. Text passed through CLIP Transformer $\rightarrow$ normalized 512-dim vector $v_t$.
  3. Vectors are concatenated: $v_{fused} = [v_i, v_t] \in \mathbb{R}^{1024}$.
  4. Normalization ensures equal geometric contribution between visual and linguistic modalities.

### Q5: What are the primary failure modes of this system?
* **Answer:**
  1. **Subtle Irony / Sarcasm:** Metaphorical memes that require deep cultural context.
  2. **OCR Noise:** Missed or misspelled text from noisy OCR in real-world scraping.
  3. **Context Drift:** Emerging meme formats and slang that were not present in CLIP's training corpus.
